# H3: Joint Fitness Model Comparison

M4 (joint W with ω + κ) vs M1 (effort-only), M2 (threat-only), M3 (single-parameter).

**Note:** SVI results shown as exploratory benchmarks. Confirmatory uses MCMC WAIC + PSIS-LOO.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, os
import matplotlib.pyplot as plt
from pathlib import Path

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10,
                     'axes.spines.right': False, 'axes.spines.top': False})

REPO_ROOT = Path(os.getcwd())
for _ in range(5):
    if (REPO_ROOT / '.git').exists(): break
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)

# Load MCMC results (CM2 models with saturating S, cell-mean vigor)
mcmc_path = Path('results/stats/joint_optimal/mcmc_model_comparison.csv')
if mcmc_path.exists():
    mc = pd.read_csv(mcmc_path)
    # M5 in the MCMC script = our M4 (joint model) in the prereg
    mc['Model_prereg'] = mc['Model'].map({
        'M1': 'M1', 'M2': 'M2', 'M3': 'M3', 'M4': 'M_sep', 'M5': 'M4_joint'
    })
    mc_main = mc[mc['Model_prereg'].isin(['M1','M2','M3','M4_joint'])].copy()
    source = 'MCMC'
    ic_col = 'WAIC'
    print(f'Loaded MCMC model comparison ({len(mc_main)} models)')
else:
    print('No MCMC results — run scripts/mcmc/run_model_comparison_mcmc.py on GPU')
    mc_main = None
    source = None

if mc_main is not None:
    m4 = mc_main[mc_main['Model_prereg']=='M4_joint'].iloc[0]
    print(f'\nJoint model (M4): choice acc={m4["choice_acc"]:.3f}, vigor r²={m4["vigor_r2"]:.3f}')
    if not np.isnan(m4.get('WAIC', np.nan)):
        print(f'  WAIC = {m4["WAIC"]:.1f}')
    print(f'\nAvailable models: {mc_main["Model_prereg"].tolist()}')
    if 'M1' not in mc_main['Model_prereg'].values:
        print('  NOTE: M1 (effort-only) not yet run — H3a pending')

Loaded MCMC model comparison (3 models)

Joint model (M4): choice acc=0.780, vigor r²=0.387
  WAIC = 12672.9

Available models: ['M2', 'M3', 'M4_joint']
  NOTE: M1 (effort-only) not yet run — H3a pending


## H3a: M4 vs M1 — Threat matters beyond effort

In [2]:
if mc_main is not None and 'M1' in mc_main['Model_prereg'].values:
    m1 = mc_main[mc_main['Model_prereg']=='M1'].iloc[0]
    if not np.isnan(m1.get('WAIC', np.nan)) and not np.isnan(m4.get('WAIC', np.nan)):
        dwaic = m1['WAIC'] - m4['WAIC']
        passed = dwaic > 0
        print('H3a — Joint vs Effort-only')
        print('=' * 50)
        print(f'  Effort-only: acc={m1["choice_acc"]:.3f}')
        print(f'  Joint:       acc={m4["choice_acc"]:.3f}')
        print(f'  ΔWAIC = {dwaic:+.1f}')
        print(f'  H3a: {"PASS ✓" if passed else "FAIL ✗"}')
    else:
        print('H3a: WAIC not available for M1 — pending GPU rerun')
else:
    print('H3a: M1 not yet fitted — pending GPU rerun')

H3a: M1 not yet fitted — pending GPU rerun


## H3b: M4 vs M2 — Individual effort differences matter

In [3]:
if mc_main is not None and 'M2' in mc_main['Model_prereg'].values:
    m2 = mc_main[mc_main['Model_prereg']=='M2'].iloc[0]
    if not np.isnan(m2.get('WAIC', np.nan)) and not np.isnan(m4.get('WAIC', np.nan)):
        dwaic = m2['WAIC'] - m4['WAIC']
        passed = dwaic > 0
        print('H3b — Joint vs Threat-only')
        print('=' * 50)
        print(f'  Threat-only: acc={m2["choice_acc"]:.3f}, vigor r²={m2["vigor_r2"]:.3f}')
        print(f'  Joint:       acc={m4["choice_acc"]:.3f}, vigor r²={m4["vigor_r2"]:.3f}')
        print(f'  ΔWAIC = {dwaic:+.1f}')
        print(f'  H3b: {"PASS ✓" if passed else "FAIL ✗"}')
    else:
        print('H3b: WAIC not available — pending')
else:
    print('H3b: M2 not fitted')

H3b — Joint vs Threat-only
  Threat-only: acc=0.789, vigor r²=0.006
  Joint:       acc=0.780, vigor r²=0.387
  ΔWAIC = +2067.3
  H3b: PASS ✓


## H3c: M4 vs M3 — Capture cost and effort cost are separable

In [4]:
if mc_main is not None and 'M3' in mc_main['Model_prereg'].values:
    m3 = mc_main[mc_main['Model_prereg']=='M3'].iloc[0]
    if not np.isnan(m3.get('WAIC', np.nan)) and not np.isnan(m4.get('WAIC', np.nan)):
        dwaic = m3['WAIC'] - m4['WAIC']
        passed = dwaic > 0
        print('H3c — Joint vs Single-parameter')
        print('=' * 50)
        print(f'  Single-param: acc={m3["choice_acc"]:.3f}, vigor r²={m3["vigor_r2"]:.3f}')
        print(f'  Joint:        acc={m4["choice_acc"]:.3f}, vigor r²={m4["vigor_r2"]:.3f}')
        print(f'  ΔWAIC = {dwaic:+.1f}')
        print(f'  H3c: {"PASS ✓" if passed else "FAIL ✗"}')
    else:
        print('H3c: WAIC not available — pending')
else:
    print('H3c: M3 not fitted')

H3c — Joint vs Single-parameter
  Single-param: acc=0.773, vigor r²=0.101
  Joint:        acc=0.780, vigor r²=0.387
  ΔWAIC = +2702.1
  H3c: PASS ✓


## Summary

In [5]:
print('H3 SUMMARY (MCMC)')
print('=' * 60)
if mc_main is not None:
    tests = []
    for alt_prereg, hname in [('M1','H3a'),('M2','H3b'),('M3','H3c')]:
        row = mc_main[mc_main['Model_prereg']==alt_prereg]
        if len(row) == 0:
            tests.append((hname, alt_prereg, 'N/A', 'N/A', 'PENDING'))
            continue
        alt_waic = row['WAIC'].values[0]
        m4_waic = m4.get('WAIC', np.nan)
        if np.isnan(alt_waic) or np.isnan(m4_waic):
            tests.append((hname, alt_prereg, 'N/A', row['choice_acc'].values[0], 'PENDING (no WAIC)'))
        else:
            dw = alt_waic - m4_waic
            p = '✓' if dw > 0 else '✗'
            tests.append((hname, alt_prereg, f'{dw:+.0f}', f'{row["choice_acc"].values[0]:.3f}', p))
    
    print(f'{"Test":<8} {"Alt":<5} {"ΔWAIC":>8} {"Alt acc":>8} {"Pass":>6}')
    print('-' * 40)
    for hname, alt, dw, acc, p in tests:
        print(f'{hname:<8} {alt:<5} {dw:>8} {acc:>8} {p:>6}')
    
    n_pass = sum(1 for *_, p in tests if p == '✓')
    n_pending = sum(1 for *_, p in tests if 'PENDING' in str(p))
    print(f'\n{n_pass}/{len(tests)} pass, {n_pending} pending.')
else:
    print('No MCMC results available.')

H3 SUMMARY (MCMC)
Test     Alt      ΔWAIC  Alt acc   Pass
----------------------------------------
H3a      M1         N/A      N/A PENDING
H3b      M2       +2067    0.789      ✓
H3c      M3       +2702    0.773      ✓

2/3 pass, 1 pending.
